# 01 — Record the sweep

Replaces the single 15 s clip in `scenario_redcar_behind_bus.ipynb` with a sweep of
**~15 configurations x 2 conditions x 1 second (20 frames)**. The scene is parked, so
a long clip is 300 near-identical frames; variety across configurations is what carries
information.

The two conditions per configuration are identical (same blueprints, same camera,
same spawn point) **except for which red car sits behind the bus**:

| condition | red target | red distractor | correct answer to *"red car behind the bus"* |
|-----------|------------|----------------|----------------------------------------------|
| `behind`  | behind the bus | off to the side | the **target** |
| `front`   | in front of the bus | behind the bus | the **distractor** |

So the same prompt has a different correct answer in each condition — that is what
`02_evaluate_grounding.ipynb` exploits with the flip test.

Run the cells top to bottom. Cell order is: setup -> helpers -> **config table (edit me)**
-> preview (no recording) -> sweep -> summary -> cleanup.

## 0. Imports and connection

In [ ]:
import carla
import math
import os
import json
import queue
import random
import shutil
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

%matplotlib inline

In [ ]:
client = carla.Client('localhost', 2000)
client.set_timeout(200.0)
world  = client.reload_world()
bp_lib = world.get_blueprint_library()
spawn_points = world.get_map().get_spawn_points()
print(len(spawn_points), 'spawn points')

## 1. Synchronous mode — 20 fps

In [ ]:
FPS = 20

settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 1.0 / FPS
world.apply_settings(settings)
print('sync mode on')

## 2. Blueprints, colours, vehicle classes

Unchanged from `scenario_redcar_behind_bus.ipynb` — same blueprints, same red, same
class rule, so the GT graphs stay schema-compatible with the ones already in
`runs/redcar_behind_bus/`.

In [ ]:
RED = '180,20,20'
WIDTH, HEIGHT, FOV = 1920, 1080, 90

COLOR_NAMES = {'180,20,20': 'red'}

def find_bp(*names):
    for n in names:
        hits = bp_lib.filter(n)
        if hits:
            return hits[0]
    raise RuntimeError(f'no blueprint matching {names}')

def vehicle_class(actor):
    wheels = int(actor.attributes.get('number_of_wheels', 4))
    if 'fusorosa' in actor.type_id or 't2' in actor.type_id:
        return 'bus'
    return 'motorcycle' if wheels == 2 else 'car'

def clear_vehicles():
    for a in world.get_actors().filter('*vehicle*'):
        a.destroy()

## 3. Projection helpers

**Verbatim from the scenario notebook — do not edit.**

In [ ]:
def build_projection_matrix(w, h, fov):
    focal = w / (2.0 * np.tan(fov * np.pi / 360.0))
    K = np.identity(3)
    K[0, 0] = K[1, 1] = focal
    K[0, 2] = w / 2.0
    K[1, 2] = h / 2.0
    return K

def get_image_point(loc, K, w2c):
    point = np.array([loc.x, loc.y, loc.z, 1])
    point_camera = np.dot(w2c, point)
    # UE4 -> standard camera coords: (x, y, z) -> (y, -z, x)
    point_camera = [point_camera[1], -point_camera[2], point_camera[0]]
    point_img = np.dot(K, point_camera)
    point_img[0] /= point_img[2]
    point_img[1] /= point_img[2]
    return point_img[0:2]

def camera_xyz(loc, w2c):
    """3D position in camera frame: x=right, y=down, z=forward (metres)."""
    p = np.dot(w2c, np.array([loc.x, loc.y, loc.z, 1]))
    return np.array([p[1], -p[2], p[0]])

K = build_projection_matrix(WIDTH, HEIGHT, FOV)

## 4. Ground-truth scene graph

Same body and **same JSON schema** as the scenario notebook — `frame` / `convention` /
`ids` / `nodes` / `edges`, nodes carry `box2d` / `class` / `color`, edges are
viewer-centric. The only change is that `ids` is now a parameter instead of a global,
because the sweep re-spawns actors for every clip.

In [ ]:
def frame_graph(w2c, frame_id, ids):
    nodes = []
    for a in world.get_actors().filter('*vehicle*'):
        tf = a.get_transform()
        cxyz = camera_xyz(tf.location, w2c)
        if cxyz[2] <= 0.1:            # behind the camera
            continue

        verts = a.bounding_box.get_world_vertices(tf)
        pts = np.array([get_image_point(v, K, w2c) for v in verts])
        x1, y1 = pts[:, 0].min(), pts[:, 1].min()
        x2, y2 = pts[:, 0].max(), pts[:, 1].max()

        raw_col = a.attributes.get('color')
        nodes.append({
            'id': int(a.id),
            'class': vehicle_class(a),
            'color': COLOR_NAMES.get(raw_col),
            'color_rgb': raw_col,
            'box2d': [float(x1), float(y1), float(x2), float(y2)],
            'loc': [tf.location.x, tf.location.y, tf.location.z],
            'yaw': tf.rotation.yaw,
            '_cam': cxyz.tolist(),
        })

    # viewer-centric relations
    edges = []
    for a in nodes:
        for b in nodes:
            if a['id'] == b['id']:
                continue
            dz = a['_cam'][2] - b['_cam'][2]    # +ve => a further away
            dx = a['_cam'][0] - b['_cam'][0]    # +ve => a to the right
            if dz >  2.0: edges.append({'subj': a['id'], 'relation': 'behind',      'obj': b['id']})
            if dz < -2.0: edges.append({'subj': a['id'], 'relation': 'in_front_of', 'obj': b['id']})
            if dx >  1.5: edges.append({'subj': a['id'], 'relation': 'right_of',    'obj': b['id']})
            if dx < -1.5: edges.append({'subj': a['id'], 'relation': 'left_of',     'obj': b['id']})

    for n in nodes:
        n.pop('_cam')

    return {'frame': frame_id, 'convention': 'viewer_centric',
            'ids': ids, 'nodes': nodes, 'edges': edges}

## 5. Config table — **this is the cell to edit**

One row per configuration. Everything downstream reads `CONFIGS`, so you can change a
number here, re-run this cell, then re-run the preview or the sweep without touching
anything else.

Fields:

- `spawn_idx` — index into `spawn_points`; the bus is parked there and everything else
  is placed in the bus's local frame.
- `lat_off` / `fwd_off` — the *side* slot for the distractor, in metres right / forward
  of the bus.
- `azimuth` — degrees off the bus's nose axis where the camera sits. Kept inside ±38°
  so the camera stays in front of the bus; that is what makes "behind the bus in the
  bus's frame" coincide with "behind the bus from the viewer's point of view".
- `elevation` — degrees above the ground plane.
- `standoff` — metres from the camera to the bus.
- Camera **roll is always 0.0**.

Disk cost: 20 frames x 2 conditions x 15 configs = 600 PNGs at 1920x1080 (~2 GB).
Drop `WIDTH, HEIGHT` in cell 2 if that is too much.

In [ ]:
SEED            = 20260731
N_CONFIGS       = 15
FRAMES_PER_CLIP = 20            # 1 second at 20 fps
BUS_GAP         = 12.0          # metres between the bus and the car in the behind/front slot
OUT_ROOT        = 'runs/sweep'
MAX_RETRIES     = 5             # camera perturbations before a config is skipped
SETTLE_TICKS    = 5             # ticks after spawning, so vehicles come to rest

CONDITIONS = ['behind', 'front']

rng = random.Random(SEED)
_spawn_choices = rng.sample(range(len(spawn_points)), N_CONFIGS)   # distinct spawn points

CONFIGS = []
for i in range(N_CONFIGS):
    CONFIGS.append({
        'config_id' : f'cfg{i:02d}',
        'spawn_idx' : _spawn_choices[i],
        'lat_off'   : rng.choice([-1.0, 1.0]) * rng.uniform(4.5, 7.5),
        'fwd_off'   : rng.uniform(2.0, 9.0),
        'azimuth'   : rng.uniform(-38.0, 38.0),
        'elevation' : rng.uniform(12.0, 30.0),
        'standoff'  : rng.uniform(26.0, 40.0),
    })

CONFIG_BY_ID = {c['config_id']: c for c in CONFIGS}

for c in CONFIGS:
    print('{config_id}  spawn={spawn_idx:>4}  lat={lat_off:+6.1f}  fwd={fwd_off:5.1f}  '
          'az={azimuth:+6.1f}  el={elevation:5.1f}  standoff={standoff:5.1f}'.format(**c))

## 6. Scene construction

Three slots in the bus's local frame. A condition just decides which red car goes in
which slot — same blueprints, same colours, same camera either way.

- `behind` slot: `BUS_GAP` m behind the bus, on its axis.
- `front`  slot: `BUS_GAP` m in front of the bus, nudged sideways by `0.35 * lat_off`
  so the near car does not sit exactly on top of the bus in image space.
- `side`   slot: `fwd_off` forward, `lat_off` right.

`ROLE_SLOT` is the whole difference between the two conditions.

In [ ]:
ROLE_SLOT = {
    'behind': {'target': 'behind', 'distractor': 'side'},
    'front' : {'target': 'front',  'distractor': 'behind'},
}

# who "red car behind the bus" actually refers to, per condition
CORRECT_ROLE  = {'behind': 'target',     'front': 'distractor'}
CONFUSER_ROLE = {'behind': 'distractor', 'front': 'target'}


class SpawnFailed(RuntimeError):
    pass


def scene_slots(anchor, cfg):
    fwd, right = anchor.get_forward_vector(), anchor.get_right_vector()

    def at(f, r):
        return carla.Transform(
            carla.Location(x=anchor.location.x + fwd.x * f + right.x * r,
                           y=anchor.location.y + fwd.y * f + right.y * r,
                           z=anchor.location.z + 0.3),
            anchor.rotation)

    return {
        'behind': at(-BUS_GAP, 0.0),
        'front' : at(+BUS_GAP, cfg['lat_off'] * 0.35),
        'side'  : at(cfg['fwd_off'], cfg['lat_off']),
    }


def spawn_scene(cfg, condition):
    """Spawn bus + red target + red distractor. Returns (actors, ids)."""
    clear_vehicles()
    anchor = spawn_points[cfg['spawn_idx']]
    slots  = scene_slots(anchor, cfg)
    actors = []

    try:
        bus_bp = find_bp('vehicle.mitsubishi.fusorosa', 'vehicle.volkswagen.t2', '*bus*')
        bus = world.try_spawn_actor(bus_bp, anchor)
        if bus is None:
            raise SpawnFailed('bus blocked at spawn point')
        actors.append(bus)

        made = {'bus': bus}
        for role in ('target', 'distractor'):
            bp = find_bp('vehicle.tesla.model3', 'vehicle.audi.a2')
            bp.set_attribute('color', RED)
            a = world.try_spawn_actor(bp, slots[ROLE_SLOT[condition][role]])
            if a is None:
                raise SpawnFailed(f'{role} blocked in {ROLE_SLOT[condition][role]} slot')
            actors.append(a)
            made[role] = a
    except Exception:
        for a in actors:
            a.destroy()
        raise

    ids = {'bus': made['bus'].id, 'target': made['target'].id,
           'distractor': made['distractor'].id}
    return actors, ids


def destroy(actors):
    for a in actors or []:
        try:
            a.destroy()
        except Exception:
            pass

## 7. Camera

Spawned **once** and re-aimed per configuration with `set_transform` — cheaper than
respawning a sensor 30 times, and it keeps a single image queue.

The camera sits `standoff` metres from the bus, `azimuth` degrees off its nose,
`elevation` degrees up, aimed back at the bus. Roll is always 0.0.

In [ ]:
cam_bp = bp_lib.find('sensor.camera.rgb')
cam_bp.set_attribute('image_size_x', str(WIDTH))
cam_bp.set_attribute('image_size_y', str(HEIGHT))
cam_bp.set_attribute('fov', str(FOV))
# NOTE: no sensor_tick -> exactly one image per world.tick()

# parked high above the map; every config re-aims it with set_transform()
_park = spawn_points[0].location
camera = world.spawn_actor(cam_bp, carla.Transform(
    carla.Location(x=_park.x, y=_park.y, z=_park.z + 60.0),
    carla.Rotation(pitch=-90.0, yaw=0.0, roll=0.0)))

image_queue = queue.Queue()
camera.listen(image_queue.put)
print('camera', camera.id)

In [ ]:
def look_at_rotation(src, dst):
    dx, dy, dz = dst.x - src.x, dst.y - src.y, dst.z - src.z
    return carla.Rotation(
        pitch=math.degrees(math.atan2(dz, math.hypot(dx, dy))),
        yaw=math.degrees(math.atan2(dy, dx)),
        roll=0.0)                                   # roll is always 0.0


def camera_transform_for(cfg, jitter=None):
    anchor = spawn_points[cfg['spawn_idx']]
    az   = cfg['azimuth']   + (jitter['azimuth']   if jitter else 0.0)
    el   = cfg['elevation'] + (jitter['elevation'] if jitter else 0.0)
    dist = cfg['standoff']  * (jitter['standoff_scale'] if jitter else 1.0)

    bearing = math.radians(anchor.rotation.yaw + az)   # bus -> camera, 0 = straight ahead
    horiz   = dist * math.cos(math.radians(el))
    height  = dist * math.sin(math.radians(el))

    src = carla.Location(x=anchor.location.x + horiz * math.cos(bearing),
                         y=anchor.location.y + horiz * math.sin(bearing),
                         z=anchor.location.z + height + 1.0)
    aim = carla.Location(x=anchor.location.x, y=anchor.location.y, z=anchor.location.z + 1.0)
    return carla.Transform(src, look_at_rotation(src, aim))


def aim_camera(cam_tf):
    camera.set_transform(cam_tf)
    world.get_spectator().set_transform(cam_tf)   # eyeball framing in the CARLA window
    for _ in range(2):                            # let the transform take effect
        world.tick()
        image_queue.get()


def capture():
    """One tick -> (rgb array, world-to-camera matrix)."""
    world.tick()
    image = image_queue.get()
    arr = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))[:, :, :3][:, :, ::-1]
    return arr, np.array(camera.get_transform().get_inverse_matrix())


def settle():
    for _ in range(SETTLE_TICKS):
        world.tick()
        image_queue.get()


def cam_record(cam_tf):
    r = cam_tf.rotation
    return {'location': [cam_tf.location.x, cam_tf.location.y, cam_tf.location.z],
            'rotation': {'pitch': r.pitch, 'yaw': r.yaw, 'roll': r.roll},
            'fov': float(FOV), 'image_size': [WIDTH, HEIGHT]}

## 8. Validation

Run on a single tick before anything is written to disk. A clip is accepted only if:

1. **In frame** — all three vehicles project in front of the camera and their full 2D
   boxes lie inside `[0, WIDTH] x [0, HEIGHT]`.
2. **Separated** — no pair of 2D boxes overlaps by more than 50 % IoU.
3. **Relation holds** — the condition-defining edges are present in the GT graph, and
   in the `behind` condition the distractor must *not* also be behind the bus (otherwise
   "red car behind the bus" has two correct answers).

In [ ]:
IOU_MAX_PAIR = 0.5

REQUIRED_EDGES = {
    'behind': [('target', 'behind', 'bus')],
    'front' : [('target', 'in_front_of', 'bus'), ('distractor', 'behind', 'bus')],
}
FORBIDDEN_EDGES = {
    'behind': [('distractor', 'behind', 'bus')],   # keeps the answer unique
    'front' : [],
}


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def has_edge(graph, ids, subj_role, relation, obj_role):
    return any(e['subj'] == ids[subj_role] and e['obj'] == ids[obj_role]
               and e['relation'] == relation
               for e in graph['edges'])


def validate_clip(graph, ids, condition):
    """Returns a list of failure reasons; empty list means the clip is good."""
    reasons = []
    by_id = {n['id']: n for n in graph['nodes']}

    missing = [r for r, i in ids.items() if i not in by_id]
    if missing:
        return [f'not projected (behind camera or culled): {", ".join(sorted(missing))}']

    for role, i in ids.items():
        x1, y1, x2, y2 = by_id[i]['box2d']
        if x1 < 0 or y1 < 0 or x2 > WIDTH or y2 > HEIGHT:
            reasons.append(f'{role} box outside image bounds '
                           f'({x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f})')

    roles = sorted(ids)
    for i in range(len(roles)):
        for j in range(i + 1, len(roles)):
            ra, rb = roles[i], roles[j]
            v = iou(by_id[ids[ra]]['box2d'], by_id[ids[rb]]['box2d'])
            if v > IOU_MAX_PAIR:
                reasons.append(f'{ra}/{rb} boxes overlap IoU={v:.2f} > {IOU_MAX_PAIR}')

    for s, rel, o in REQUIRED_EDGES[condition]:
        if not has_edge(graph, ids, s, rel, o):
            reasons.append(f'missing relation edge {s} {rel} {o}')
    for s, rel, o in FORBIDDEN_EDGES[condition]:
        if has_edge(graph, ids, s, rel, o):
            reasons.append(f'ambiguous: {s} is also {rel} {o}')

    return reasons


def probe(cfg, condition):
    """Spawn, settle, grab one frame + graph, tear down. No files written."""
    actors = None
    try:
        actors, ids = spawn_scene(cfg, condition)
        settle()
        arr, w2c = capture()
        graph = frame_graph(w2c, 0, ids)
        return arr, graph, ids, validate_clip(graph, ids, condition)
    finally:
        destroy(actors)

## 9. Preview — render one config + condition inline, **without recording**

Edit the two constants, re-run. Green = target, orange = distractor, blue = bus.
The validation verdict for that exact framing is printed underneath, so you can iterate
on `azimuth` / `elevation` / `standoff` in the config cell until you like the shot.

In [ ]:
PREVIEW_CONFIG    = 'cfg00'
PREVIEW_CONDITION = 'behind'        # 'behind' or 'front'

ROLE_COLOR = {'target': '#20c020', 'distractor': '#ff9500', 'bus': '#3080ff'}


def draw_gt(ax, graph, ids, lw=2.5):
    role_of = {v: k for k, v in ids.items()}
    for n in graph['nodes']:
        role = role_of.get(n['id'])
        if role is None:
            continue
        x1, y1, x2, y2 = n['box2d']
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor=ROLE_COLOR[role], linewidth=lw))
        ax.text(x1, y1 - 6, f"{role}:{n['id']} {n['color'] or ''} {n['class']}",
                color=ROLE_COLOR[role], fontsize=9, weight='bold')


cfg = CONFIG_BY_ID[PREVIEW_CONFIG]
aim_camera(camera_transform_for(cfg))
arr, graph, ids, reasons = probe(cfg, PREVIEW_CONDITION)

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(arr)
draw_gt(ax, graph, ids)
ax.set_title(f'{PREVIEW_CONFIG} / {PREVIEW_CONDITION}   '
             f'(correct answer = {CORRECT_ROLE[PREVIEW_CONDITION]})')
ax.axis('off')
plt.show()

print('ids            :', ids)
print('correct answer :', CORRECT_ROLE[PREVIEW_CONDITION],
      '->', ids[CORRECT_ROLE[PREVIEW_CONDITION]])
print('edges (target/distractor vs bus):')
for e in graph['edges']:
    if e['obj'] == ids['bus'] and e['subj'] != ids['bus']:
        print('   ', {v: k for k, v in ids.items()}[e['subj']], e['relation'], 'bus')
print('validation     :', 'OK' if not reasons else 'FAIL')
for r in reasons:
    print('   -', r)

## 10. Sweep

For each configuration: build a camera transform, probe **both** conditions with it,
and only accept if both pass validation — that is what keeps the camera identical
across the two conditions of a config. On failure the camera is perturbed (azimuth,
elevation, standoff; roll stays 0.0) and retried up to `MAX_RETRIES` times; after that
the config is skipped and the reasons are logged.

Every spawn is torn down in a `finally`, so a failure mid-config cannot leak actors.

Re-running this cell overwrites `runs/sweep`.

In [ ]:
def record_clip(cfg, condition, cam_tf):
    """Spawn, write FRAMES_PER_CLIP frames of RGB + GT graph, tear down."""
    base = os.path.join(OUT_ROOT, cfg['config_id'], condition)
    rgb_dir, gt_dir = os.path.join(base, 'rgb'), os.path.join(base, 'gt_graphs')
    os.makedirs(rgb_dir, exist_ok=True)
    os.makedirs(gt_dir, exist_ok=True)

    actors = None
    try:
        actors, ids = spawn_scene(cfg, condition)
        settle()
        for i in range(FRAMES_PER_CLIP):
            arr, w2c = capture()
            Image.fromarray(arr).save(os.path.join(rgb_dir, f'{i:06d}.png'))
            with open(os.path.join(gt_dir, f'{i:06d}.json'), 'w') as f:
                json.dump(frame_graph(w2c, i, ids), f)
    finally:
        destroy(actors)

    return ids, rgb_dir.replace('\\', '/'), gt_dir.replace('\\', '/')


def jitter_for(attempt, r):
    if attempt == 0:
        return None
    return {'azimuth'       : r.uniform(-14.0, 14.0),
            'elevation'     : r.uniform(-4.0, 9.0),
            'standoff_scale': 1.0 + 0.12 * attempt}

In [ ]:
if os.path.isdir(OUT_ROOT):
    shutil.rmtree(OUT_ROOT)
os.makedirs(OUT_ROOT, exist_ok=True)

manifest = {
    'seed': SEED,
    'fps': FPS,
    'frames_per_clip': FRAMES_PER_CLIP,
    'conditions': CONDITIONS,
    'prompt': 'red car behind the bus',
    'clips': [],
    'skipped': [],
}

for cfg in CONFIGS:
    cid = cfg['config_id']
    jr = random.Random(f'{SEED}-{cid}')          # deterministic per-config perturbations
    accepted, attempts_log = None, []

    try:
        for attempt in range(MAX_RETRIES + 1):   # 1 initial framing + MAX_RETRIES retries
            cam_tf = camera_transform_for(cfg, jitter_for(attempt, jr))
            aim_camera(cam_tf)

            failures = {}
            for cond in CONDITIONS:
                _, _, _, reasons = probe(cfg, cond)
                if reasons:
                    failures[cond] = reasons

            if not failures:
                accepted = (cam_tf, attempt)
                break
            attempts_log.append({'attempt': attempt, 'failures': failures})
            print(f'  {cid} attempt {attempt}: '
                  + '; '.join(f'[{c}] ' + ' | '.join(r) for c, r in failures.items()))

    except SpawnFailed as e:
        manifest['skipped'].append({'config_id': cid, 'reason': f'spawn failed: {e}',
                                    'attempts': len(attempts_log) + 1})
        print(f'{cid}: SKIP -- spawn failed: {e}')
        continue

    if accepted is None:
        manifest['skipped'].append({
            'config_id': cid,
            'reason': f'validation failed after {MAX_RETRIES} camera perturbations',
            'attempts': len(attempts_log),
            'last_failures': attempts_log[-1]['failures'] if attempts_log else {},
        })
        print(f'{cid}: SKIP -- no valid framing after {MAX_RETRIES} retries')
        continue

    cam_tf, attempt = accepted
    aim_camera(cam_tf)
    for cond in CONDITIONS:
        ids, rgb_dir, gt_dir = record_clip(cfg, cond, cam_tf)
        manifest['clips'].append({
            'config_id': cid,
            'condition': cond,
            'ids': ids,
            'correct_answer_role': CORRECT_ROLE[cond],
            'correct_answer_id'  : ids[CORRECT_ROLE[cond]],
            'confuser_role'      : CONFUSER_ROLE[cond],
            'confuser_id'        : ids[CONFUSER_ROLE[cond]],
            'camera': cam_record(cam_tf),
            'frames': FRAMES_PER_CLIP,
            'rgb_dir': rgb_dir,
            'gt_dir' : gt_dir,
            'validation': {'status': 'ok', 'attempts': attempt + 1,
                           'checks': ['in_frame', 'pairwise_iou<=0.5', 'relation_edge']},
            'config': cfg,
        })
    print(f'{cid}: recorded both conditions (camera attempt {attempt + 1})')

with open(os.path.join(OUT_ROOT, 'manifest.json'), 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'\ndone -> {OUT_ROOT}/manifest.json  '
      f'({len(manifest["clips"])} clips, {len(manifest["skipped"])} configs skipped)')

## 11. Summary

In [ ]:
man = json.load(open(os.path.join(OUT_ROOT, 'manifest.json')))

n_cfg_ok = len({c['config_id'] for c in man['clips']})
print(f'configs recorded : {n_cfg_ok} / {len(CONFIGS)}')
print(f'clips recorded   : {len(man["clips"])}  ({FRAMES_PER_CLIP} frames each)')
print(f'configs skipped  : {len(man["skipped"])}')

retries = [c['validation']['attempts'] for c in man['clips']]
if retries:
    print(f'camera attempts  : mean {np.mean(retries):.2f}, max {max(retries)}')

if man['skipped']:
    print('\nskipped:')
    for s in man['skipped']:
        print(f'  {s["config_id"]}: {s["reason"]}')
        for cond, reasons in (s.get('last_failures') or {}).items():
            for r in reasons:
                print(f'      [{cond}] {r}')

nbytes = sum(os.path.getsize(os.path.join(dp, f))
             for dp, _, fs in os.walk(OUT_ROOT) for f in fs)
print(f'\non disk: {nbytes / 1e9:.2f} GB in {OUT_ROOT}')

## 12. Cleanup

Destroys the camera and every vehicle and puts the server back into async mode.
Safe to re-run.

In [ ]:
try:
    camera.stop()
    camera.destroy()
except Exception as e:
    print('camera already gone:', e)

clear_vehicles()

s = world.get_settings()
s.synchronous_mode = False
s.fixed_delta_seconds = None
world.apply_settings(s)
print('cleaned up')